
# FINN Speaker Recognition — KV260 PYNQ Deployment

This notebook runs on the **Kria KV260** with PYNQ 3.0.1 installed.It loads FINN-generated accelerator bitstreams and evaluates them againstthe MFCC test dataset, measuring accuracy, latency, and throughput.

## 1. Imports and Configuration

In [22]:
import numpy as np
import json
import time
import os
from pathlib import Path
import pynq
from pynq import Overlay, allocate

print(f'PYNQ version: {pynq.__version__}')
print(f'Board: {os.uname().nodename}')

PYNQ version: 3.0.1
Board: kria


In [23]:
# ── Paths ─────────────────────────────────────────────────────
BASE_DIR = Path('.')
TEST_DATA_DIR = BASE_DIR / 'test_data'

# Model configurations
# Both models have the same input format but different output widths
MODELS = {
    'QAT-4': {
        'bitfile': str(BASE_DIR / 'finn_build' / 'qat_4bit' / 'bitfile' / 'finn-accel.bit'),
        'out_dtype': np.int16,       # 16-bit output
        'out_bytes_per_val': 2,      # 2 bytes per output value
        'out_stream_width': 16,      # m_axis width in bits
        'weight_bits': 4,
    },
    'QAT-8': {
        'bitfile': str(BASE_DIR / 'finn_build' /'qat_8bit' / 'bitfile' / 'finn-accel.bit'),
        'out_dtype': np.int32,       # 24-bit padded to 32-bit for DMA        
        'out_bytes_per_val': 4,      # 4 bytes per output value (24-bit padded)        
        'out_stream_width': 24,      # m_axis width in bits        
        'weight_bits': 8,    },
}
# Common I/O parameters (same for both models)
INPUT_SHAPE = (20, 64, 2)     # H, W, C — MFCC shape
INPUT_DTYPE = np.int8          # INT8 input quantization
IN_STREAM_WIDTH = 16           # s_axis width in bits (2 × INT8)
NUM_CLASSES = 10

# Derived sizes
INPUT_BYTES = int(np.prod(INPUT_SHAPE))  # 20 × 64 × 2 = 2560 bytes
OUTPUT_VALS = NUM_CLASSES                # 10 output values

print(f'Input: {INPUT_SHAPE} as {INPUT_DTYPE.__name__} = {INPUT_BYTES} bytes')
print(f'Output: {OUTPUT_VALS} values')

for name, cfg in MODELS.items():    
    print(f'  {name}: out_dtype={cfg["out_dtype"].__name__}, '          
          f'stream={cfg["out_stream_width"]}b, bitfile={cfg["bitfile"]}')


Input: (20, 64, 2) as int8 = 2560 bytes
Output: 10 values
  QAT-4: out_dtype=int16, stream=16b, bitfile=finn_build/qat_4bit/bitfile/finn-accel.bit
  QAT-8: out_dtype=int32, stream=24b, bitfile=finn_build/qat_8bit/bitfile/finn-accel.bit


## 2. Load Test Data

In [24]:
# Load the numpy test set (converted from TF batches on the host)
X_test = np.load(TEST_DATA_DIR / 'X_test.npy')   # float32, (N, 20, 64, 2)
y_test = np.load(TEST_DATA_DIR / 'y_test.npy')   # int64, (N,)

with open(TEST_DATA_DIR / 'metadata.json') as f:    
    metadata = json.load(f)

class_names = metadata.get('subfolders', [str(i) for i in range(NUM_CLASSES)])
print(f'Test samples: {X_test.shape[0]}')
print(f'Input shape:  {X_test.shape[1:]}')
print(f'Classes:      {class_names}')
print(f'Value range:  [{X_test.min():.2f}, {X_test.max():.2f}]')


Test samples: 493
Input shape:  (20, 64, 2)
Classes:      ['f0001', 'f0002', 'f0003', 'f0004', 'f0005', 'm0001', 'm0002', 'm0003', 'm0004', 'm0005']
Value range:  [-5.40, 7.89]


## 3. Input Quantization

The FINN accelerator expects **INT8** input. The MFCCs are normalized float32(mean~0, std~1 per coefficient). We quantize using symmetric per-tensor scaling:`x_int8 = round(x_float * scale)` where `scale = 127 / percentile_99.9(|x|)`This matches Brevitas `Int8ActPerTensorFloat` behavior — FINN absorbs the scalefactor into the first layer's thresholds during compilation.

In [25]:
def quantize_input(x_float, scale=None):
    """Quantize float32 MFCCs to INT8 for the FINN accelerator.

    Parameters
    ----------
    x_float : np.ndarray, shape (N, 20, 64, 2), dtype float32
    scale   : float, optional. If None, computed from the data.

    Returns
    -------
    x_int8  : np.ndarray, same shape, dtype int8
    scale   : float, the scale factor used
    """
    if scale is None:
        # Use 99.9th percentile to avoid outlier-driven clipping
        abs_max = np.percentile(np.abs(x_float), 99.9)
        scale = 127.0 / (abs_max + 1e-8)

    x_scaled = np.round(x_float * scale).astype(np.int32)
    x_int8 = np.clip(x_scaled, -128, 127).astype(np.int8)
    return x_int8, scale

# Quantize the entire test set
X_test_int8, quant_scale = quantize_input(X_test)

print(f'Quantization scale: {quant_scale:.4f}')
print(f'INT8 range used: [{X_test_int8.min()}, {X_test_int8.max()}]')
print(f'Clipped values: {np.sum(np.abs(X_test_int8) == 127) + np.sum(np.abs(X_test_int8) == 128)}')
print(f'Shape: {X_test_int8.shape}, dtype: {X_test_int8.dtype}')

Quantization scale: 35.1088
INT8 range used: [-128, 127]
Clipped values: 825
Shape: (493, 20, 64, 2), dtype: int8


## 4. DMA Transfer Functions

FINN's accelerator uses AXI-Stream with IODMA:- **Input DMA (MM2S):** Sends packed INT8 data from DDR to the accelerator- **Output DMA (S2MM):** Receives classification logits from the acceleratorThe input is packed as raw bytes — 2 INT8 values per 16-bit AXI beat (I and Qchannels at each spatial position). The output format depends on the model:- QAT-4: 10 × INT16 values (20 bytes)- QAT-8: 10 × INT24 values padded to 32-bit (40 bytes)

In [26]:
def run_single_inference(idma, odma, x_sample_int8, model_cfg):
    """Run a single inference through the FINN accelerator.

    FINN uses HLS-based IODMAs with register-level control:
      idma0: in0_V_1/in0_V_2 (64-bit phys addr), numReps, CTRL.AP_START
      odma0: out_V_1/out_V_2 (64-bit phys addr), numReps, CTRL.AP_START
    """
    out_bytes = OUTPUT_VALS * model_cfg['out_bytes_per_val']

    # Allocate physically contiguous DMA buffers
    ibuf = allocate(shape=(INPUT_BYTES,), dtype=np.uint8)
    obuf = allocate(shape=(out_bytes,), dtype=np.uint8)

    # Pack input
    ibuf[:] = x_sample_int8.flatten().view(np.uint8)
    obuf[:] = 0

    # Get physical addresses
    in_addr = ibuf.device_address
    out_addr = obuf.device_address

    # Program output DMA first (must be ready to receive)
    odma.register_map.out_V_1 = out_addr & 0xFFFFFFFF
    odma.register_map.out_V_2 = (out_addr >> 32) & 0xFFFFFFFF
    odma.register_map.numReps = 1

    # Program input DMA
    idma.register_map.in0_V_1 = in_addr & 0xFFFFFFFF
    idma.register_map.in0_V_2 = (in_addr >> 32) & 0xFFFFFFFF
    idma.register_map.numReps = 1

    # Start both DMAs (odma first so it's ready to receive)
    odma.register_map.CTRL.AP_START = 1
    idma.register_map.CTRL.AP_START = 1

    # Poll for completion (wait for both to finish)
    while not odma.register_map.CTRL.AP_DONE:
        pass

    # Unpack output
    if model_cfg['out_stream_width'] == 16:
        logits = np.frombuffer(obuf, dtype=np.int16)[:OUTPUT_VALS].astype(np.float32)
    elif model_cfg['out_stream_width'] == 24:
        raw32 = np.frombuffer(obuf, dtype=np.int32)[:OUTPUT_VALS]
        logits = np.where(raw32 & 0x800000, raw32 | np.int32(0xFF000000),
                          raw32).astype(np.float32)
    else:
        logits = np.frombuffer(obuf, dtype=model_cfg['out_dtype'])[:OUTPUT_VALS].astype(np.float32)

    ibuf.freebuffer()
    obuf.freebuffer()

    return logits


def run_batch_inference(idma, odma, X_batch_int8, model_cfg, verbose=True):
    """Run inference on a batch of samples, collecting predictions and timing."""
    n_samples = X_batch_int8.shape[0]
    predictions = np.zeros(n_samples, dtype=np.int64)
    latencies = np.zeros(n_samples, dtype=np.float64)

    out_bytes = OUTPUT_VALS * model_cfg['out_bytes_per_val']

    # Pre-allocate persistent buffers
    ibuf = allocate(shape=(INPUT_BYTES,), dtype=np.uint8)
    obuf = allocate(shape=(out_bytes,), dtype=np.uint8)

    in_addr = ibuf.device_address
    out_addr = obuf.device_address

    for i in range(n_samples):
        # Pack input
        ibuf[:] = X_batch_int8[i].flatten().view(np.uint8)
        obuf[:] = 0

        # ── Timed inference ──────────────────────────────
        t0 = time.perf_counter()

        # Program ODma (receiver) first
        odma.register_map.out_V_1 = out_addr & 0xFFFFFFFF
        odma.register_map.out_V_2 = (out_addr >> 32) & 0xFFFFFFFF
        odma.register_map.numReps = 1

        # Program IDma (sender)
        idma.register_map.in0_V_1 = in_addr & 0xFFFFFFFF
        idma.register_map.in0_V_2 = (in_addr >> 32) & 0xFFFFFFFF
        idma.register_map.numReps = 1

        # Start both
        odma.register_map.CTRL.AP_START = 1
        idma.register_map.CTRL.AP_START = 1

        # Wait for output DMA to complete
        while not odma.register_map.CTRL.AP_DONE:
            pass

        t1 = time.perf_counter()
        # ─────────────────────────────────────────────────

        # Unpack and argmax
        if model_cfg['out_stream_width'] == 16:
            logits = np.frombuffer(obuf, dtype=np.int16)[:OUTPUT_VALS]
        elif model_cfg['out_stream_width'] == 24:
            raw32 = np.frombuffer(obuf, dtype=np.int32)[:OUTPUT_VALS]
            logits = np.where(raw32 & 0x800000, raw32 | np.int32(0xFF000000), raw32)
        else:
            logits = np.frombuffer(obuf, dtype=model_cfg['out_dtype'])[:OUTPUT_VALS]

        predictions[i] = np.argmax(logits)
        latencies[i] = t1 - t0

        if verbose and (i + 1) % 100 == 0:
            print(f'  {i+1}/{n_samples} samples processed...')

    ibuf.freebuffer()
    obuf.freebuffer()

    return predictions, latencies

print('FINN IODMA functions defined.')

FINN IODMA functions defined.


## 5. Run Inference and Measure Metrics
For each model (QAT-4 and QAT-8), we:
1. Load the FINN overlay (bitstream)
2. Run all test samples through the accelerator
3. Measure per-sample latency
4. Compute accuracy, throughput, and average latency

In [20]:
all_results = {}

for model_name, model_cfg in MODELS.items():
    bitfile = model_cfg['bitfile']
    if not os.path.exists(bitfile):
        print(f'[{model_name}] Bitfile not found: {bitfile} — skipping')
        continue

    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {model_name} — Loading overlay')
    print(sep)

    # Load the FINN overlay
    t_load = time.perf_counter()
    ol = Overlay(bitfile)
    t_load = time.perf_counter() - t_load
    print(f'  Overlay loaded in {t_load:.2f}s')

    # Access FINN IODMAs
    idma = ol.idma0
    odma = ol.odma0
    print(f'  idma0: {idma.register_map.CTRL}')
    print(f'  odma0: {odma.register_map.CTRL}')

    # Warmup — run a few inferences to stabilize timing
    print(f'  Warmup (5 samples)...')
    for w in range(min(5, X_test_int8.shape[0])):
        _ = run_single_inference(idma, odma, X_test_int8[w], model_cfg)

    # Run inference on all test samples
    print(f'  Running inference on {X_test_int8.shape[0]} samples...')
    preds, latencies = run_batch_inference(idma, odma, X_test_int8, model_cfg, verbose=True)

    # Compute metrics
    accuracy = np.mean(preds == y_test)
    avg_latency_us = np.mean(latencies) * 1e6
    median_latency_us = np.median(latencies) * 1e6
    min_latency_us = np.min(latencies) * 1e6
    throughput = 1.0 / np.mean(latencies)

    # Compute cycles (at 200 MHz = 5ns per cycle)
    avg_latency_cycles = int(avg_latency_us * 200)

    result = {
        'accuracy': accuracy,
        'avg_latency_us': avg_latency_us,
        'median_latency_us': median_latency_us,
        'min_latency_us': min_latency_us,
        'avg_latency_cycles': avg_latency_cycles,
        'throughput_fps': throughput,
        'n_samples': len(preds),
        'predictions': preds,
        'latencies': latencies,
        'overlay_load_time': t_load,
    }
    all_results[model_name] = result

    print(f'\n  Results:')
    print(f'    Accuracy:          {accuracy * 100:.2f}%')
    print(f'    Avg latency:       {avg_latency_us:.1f} us ({avg_latency_cycles:,} cycles)')
    print(f'    Median latency:    {median_latency_us:.1f} us')
    print(f'    Min latency:       {min_latency_us:.1f} us')
    print(f'    Throughput:        {throughput:,.0f} inferences/sec')

    # Free the overlay before loading the next one
    ol.free()
    print(f'  Overlay freed.')

print('\nAll models evaluated.')


  QAT-4 — Loading overlay
  Overlay loaded in 1.09s
  idma0: 0x4
  odma0: 0x4
  Warmup (5 samples)...


KeyboardInterrupt: 

## 6. Confusion Matrices

In [ ]:
all_results = {}

for model_name, model_cfg in MODELS.items():
    bitfile = model_cfg['bitfile']
    if not os.path.exists(bitfile):
        print(f'[{model_name}] Bitfile not found: {bitfile} — skipping')
        continue

    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  {model_name} — Loading overlay')
    print(sep)

    # Load the FINN overlay
    t_load = time.perf_counter()
    ol = Overlay(bitfile)
    t_load = time.perf_counter() - t_load
    print(f'  Overlay loaded in {t_load:.2f}s')

    # Find the DMA controller
    dma = ol.axi_dma_0
    print(f'  DMA: {dma}')

    # Run inference on all test samples
    print(f'  Running inference on {X_test_int8.shape[0]} samples...')
    preds, latencies = run_batch_inference(dma, X_test_int8, model_cfg, verbose=True)

    # Compute metrics
    accuracy = np.mean(preds == y_test)
    avg_latency_us = np.mean(latencies) * 1e6
    median_latency_us = np.median(latencies) * 1e6
    min_latency_us = np.min(latencies) * 1e6
    throughput = 1.0 / np.mean(latencies)

    # Compute cycles (at 200 MHz = 5ns per cycle)
    avg_latency_cycles = int(avg_latency_us * 200)  # 200 cycles per microsecond

    result = {
        'accuracy': accuracy,
        'avg_latency_us': avg_latency_us,
        'median_latency_us': median_latency_us,
        'min_latency_us': min_latency_us,
        'avg_latency_cycles': avg_latency_cycles,
        'throughput_fps': throughput,
        'n_samples': len(preds),
        'predictions': preds,
        'latencies': latencies,
        'overlay_load_time': t_load,
    }
    all_results[model_name] = result

    print(f'\n  Results:')
    print(f'    Accuracy:          {accuracy * 100:.2f}%')
    print(f'    Avg latency:       {avg_latency_us:.1f} us ({avg_latency_cycles:,} cycles)')
    print(f'    Median latency:    {median_latency_us:.1f} us')
    print(f'    Min latency:       {min_latency_us:.1f} us')
    print(f'    Throughput:        {throughput:,.0f} inferences/sec')

    # Free the overlay before loading the next one
    ol.free()
    print(f'  Overlay freed.')

print('\nAll models evaluated.')

## 7. Latency Distribution

In [ ]:
if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, len(all_results), figsize=(7 * len(all_results), 5))
    if len(all_results) == 1:
        axes = [axes]

    for ax, (model_name, result) in zip(axes, all_results.items()):
        lat_us = result['latencies'] * 1e6
        ax.hist(lat_us, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
        ax.axvline(np.mean(lat_us), color='red', linestyle='--',
                   label=f'Mean: {np.mean(lat_us):.1f} us')
        ax.axvline(np.median(lat_us), color='orange', linestyle='--',
                   label=f'Median: {np.median(lat_us):.1f} us')
        ax.set_title(f'{model_name} — Inference Latency')
        ax.set_xlabel('Latency (us)')
        ax.set_ylabel('Count')
        ax.legend()

    plt.tight_layout()
    plt.savefig('latency_distribution.png', dpi=150)
    plt.show()

## 8. Resource Utilization
Read from the FINN synthesis reports (generated on the host).Copy the `report/` directories from your FINN build to the Kria if you wantthese displayed here, or just reference the numbers from the host.

In [ ]:
# KV260 xck26 available resources
KV260 = {
    'LUT':      117120,
    'FF':       234240,
    'BRAM_36K': 144,
    'BRAM_18K': 288,
    'DSP':      1248,
    'URAM':     64,
}

# Post-synthesis numbers from FINN reports (paste from host if not on Kria)
# These come from post_synth_resources.json "(top)" entry
SYNTH_RESOURCES = {
    'QAT-4': {'LUT': 28041, 'FF': 30608, 'BRAM_36K': 6, 'BRAM_18K': 3, 'DSP': 34, 'URAM': 0},
    'QAT-8': {'LUT': 45187, 'FF': 28398, 'BRAM_36K': 3, 'BRAM_18K': 4, 'DSP': 32, 'URAM': 1},
}

# FINN performance estimates
PERF_ESTIMATES = {
    'QAT-4': {'est_throughput_fps': 13021, 'est_latency_ns': 361325, 'critical_path_cycles': 72265, 'bottleneck': 'MVAU_hls_1'},
    'QAT-8': {'est_throughput_fps': 13021, 'est_latency_ns': 367725, 'critical_path_cycles': 73545, 'bottleneck': 'MVAU_hls_0'},
}

print('Resource and performance data loaded.')

## 9. Final Comparison Table

In [ ]:
sep = '=' * 80
dash = '-' * 80

print(f'\n{sep}')
print('  FINN Speaker Recognition — KV260 Deployment Results'.center(80))
print(f'{sep}')

# Header
tags = list(all_results.keys())
hdr = f'  {"Metric":<32}'
for tag in tags:
    hdr += f' {tag:>20}'
print(hdr)
print(f'  {dash}')

# ── Accuracy ──
row = f'  {"Test Accuracy":<32}'
for tag in tags:
    acc = all_results[tag]['accuracy']
    row += f' {acc*100:>18.2f} %'
print(row)

# ── Latency (measured on Kria) ──
for metric, key, unit in [
    ('Avg Latency', 'avg_latency_us', 'us'),
    ('Median Latency', 'median_latency_us', 'us'),
    ('Min Latency', 'min_latency_us', 'us'),
    ('Latency (cycles)', 'avg_latency_cycles', 'cyc'),
]:
    row = f'  {metric:<32}'
    for tag in tags:
        v = all_results[tag][key]
        if unit == 'cyc':
            row += f' {v:>17,} {unit}'
        else:
            row += f' {v:>17.1f} {unit}'
    print(row)

# ── Throughput (measured) ──
row = f'  {"Throughput (measured)":<32}'
for tag in tags:
    tp = all_results[tag]['throughput_fps']
    row += f' {tp:>16,.0f} fps'
print(row)

# ── FINN estimated throughput ──
row = f'  {"Throughput (FINN est.)":<32}'
for tag in tags:
    tp = PERF_ESTIMATES.get(tag, {}).get('est_throughput_fps', 0)
    row += f' {tp:>16,.0f} fps'
print(row)

# ── FINN estimated latency ──
row = f'  {"Latency (FINN est.)":<32}'
for tag in tags:
    lat = PERF_ESTIMATES.get(tag, {}).get('est_latency_ns', 0)
    row += f' {lat/1000:>17.1f} us'
print(row)

print(f'  {dash}')

# ── Resource utilization ──
print(f'  {"FPGA RESOURCE UTILIZATION":}')
print(f'  {dash}')

for rname in ['LUT', 'FF', 'BRAM_36K', 'BRAM_18K', 'DSP', 'URAM']:
    row = f'  {rname:<32}'
    for tag in tags:
        used = SYNTH_RESOURCES.get(tag, {}).get(rname, 0)
        avail = KV260.get(rname, 1)
        pct = used / avail * 100
        row += f' {used:>10,} ({pct:>5.1f}%)'
    print(row)

print(f'  {dash}')

# ── Bottleneck layer ──
row = f'  {"Bottleneck layer":<32}'
for tag in tags:
    bn = PERF_ESTIMATES.get(tag, {}).get('bottleneck', 'N/A')
    row += f' {bn:>20}'
print(row)

print(f'\n  Target: xck26-sfvc784-2LV-c | 200 MHz | VIVADO_ZYNQ')
print(f'  Architecture: BRAM-Opt v2 (8 filters, ~1,760 params)')
print(f'{sep}')

## 10. Save Results

In [ ]:
# Save results to JSON for the paper
output = {}
for model_name, result in all_results.items():
    output[model_name] = {
        'accuracy': float(result['accuracy']),
        'avg_latency_us': float(result['avg_latency_us']),
        'median_latency_us': float(result['median_latency_us']),
        'min_latency_us': float(result['min_latency_us']),
        'avg_latency_cycles': int(result['avg_latency_cycles']),
        'throughput_fps': float(result['throughput_fps']),
        'n_test_samples': int(result['n_samples']),
        'resources': SYNTH_RESOURCES.get(model_name, {}),
        'finn_estimates': PERF_ESTIMATES.get(model_name, {}),
    }

results_path = 'deployment_results.json'
with open(results_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {results_path}')

# Also save predictions for later analysis
for model_name, result in all_results.items():
    np.save(f'predictions_{model_name.lower().replace("-","_")}.npy', result['predictions'])
    np.save(f'latencies_{model_name.lower().replace("-","_")}.npy', result['latencies'])
print('Predictions and latencies saved as .npy files.')